# 04b Taiwan Population Features

Collect and align Taiwan township population data to DeepSolar feature names.

**Data year**: 民國 109 年（2020）  
**Geographic unit**: 368 鄉鎮市區 (TOWNCODE)  
**Sources**:
- `民國97-114年各縣市鄉鎮市區土地面積及人口密度.xls` (Sheet 109) — population, land area, density
- `民國109年常住人口之年齡結構/*.xlsx` — age structure (22 county files)
- `民國109年住戶數、常住人口數及平均每戶人口數/*.xlsx` — household size (22 county files)

**Output**: `data/taiwan/population/taiwan_population_features.csv` (368 × 17)

## DeepSolar Column Mapping

| DeepSolar Column | Taiwan Source | Note |
|---|---|---|
| `population` | XLS col 1 | direct |
| `population_density` | XLS col 3 | direct (persons/km²) |
| `land_area_km2` | XLS col 2 | auxiliary — not a DeepSolar column |
| `total_area` | XLS col 2 ÷ 2.58999 | km² → sq. miles; US includes water, Taiwan land-only |
| `age_35_44_rate` ~ `age_55_64_rate` | age xlsx col 6,8,9 | direct |
| `age_median` | age xlsx col 11 | average age as proxy |
| `age_5_9_rate`, `age_10_14_rate` | age xlsx <15 group | uniform split: ×(5/15) |
| `age_15_17_rate` | age xlsx 15-24 group | uniform split: ×(3/10) |
| `age_65_74_rate` | age xlsx 65+ group | ×0.60 (Taiwan 65+ distribution) |
| `age_more_than_85_rate` | age xlsx 65+ group | ×0.08 |
| `average_household_size` | household xlsx col 4 | direct |

## Step 0 — Setup

In [ ]:
import sys
sys.path.append('../')

from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR    = Path('../data/taiwan')
POP_DIR     = DATA_DIR / 'population'  # XLS + xlsx subdirs live here
XLS_PATH    = POP_DIR / '民國97-114年各縣市鄉鎮市區土地面積及人口密度.xls'
AGE_DIR     = POP_DIR / '民國109年常住人口之年齡結構'
HH_DIR      = POP_DIR / '民國109年住戶數、常住人口數及平均每戶人口數'
CLIMATE_CSV = DATA_DIR / 'climate' / 'taiwan_climate_annual.csv'
OUT_PATH    = POP_DIR / 'taiwan_population_features.csv'

print('XLS_PATH exists:', XLS_PATH.exists())
print('AGE_DIR  exists:', AGE_DIR.exists())
print('HH_DIR   exists:', HH_DIR.exists())
print('CLIMATE  exists:', CLIMATE_CSV.exists())

## Helper Functions

In [ ]:
def _strip_ideographic(s):
    return str(s).replace('\u3000', '').strip() if pd.notna(s) else ''


def _lead_spaces(s):
    count = 0
    for c in (str(s) if pd.notna(s) else ''):
        if c == '\u3000':
            count += 1
        else:
            break
    return count


def _is_num(v):
    if pd.isna(v):
        return False
    try:
        float(v)
        return True
    except (TypeError, ValueError):
        return False


def _normalize_name(s):
    return _strip_ideographic(s).replace('台', '臺')


def _normalize_xls_name(s):
    if not pd.notna(s):
        return ''
    return str(s).replace('\u3000', '').replace(' ', '').replace('台', '臺')


_KNOWN_COUNTIES = {
    '臺北市', '新北市', '桃園市', '臺中市', '臺南市', '高雄市',
    '基隆市', '新竹市', '嘉義市',
    '宜蘭縣', '新竹縣', '苗栗縣', '彰化縣', '南投縣', '雲林縣',
    '嘉義縣', '屏東縣', '臺東縣', '花蓮縣', '澎湖縣', '連江縣', '金門縣',
}
_SKIP_NAMES = {'總計', '臺灣省', '福建省', ''}

print('Helpers defined.')

## Step 1 — Load Population Density XLS

Reads `民國97-114年各縣市鄉鎮市區土地面積及人口密度.xls`, Sheet 109.

This file covers all 368 townships including remote/indigenous ones that may be
missing from per-county XLSX files.

In [ ]:
def _load_pop_density_xls(xls_path, sheet='109'):
    """
    Parse the multi-year township land-area and population-density XLS.

    Columns (0-indexed):
      0: name (county = ASCII-spaced chars; township = leading U+3000)
      1: annual resident population
      2: land area (km²)
      3: population density (persons/km²)
    """
    df = pd.read_excel(xls_path, sheet_name=sheet, header=None)
    records, county = [], None
    for _, row in df.iterrows():
        n = row.get(0, np.nan)
        lv = _lead_spaces(n)
        if not _is_num(row.get(1)):
            continue
        cleaned = _normalize_xls_name(n).lstrip('※')
        if lv == 0:
            if cleaned in _KNOWN_COUNTIES:
                county = cleaned
            elif cleaned not in _SKIP_NAMES and county:
                records.append({
                    'county_zh': county,
                    'town_zh': cleaned,
                    'population': int(float(row[1])),
                    'land_area_km2': float(row[2]) if _is_num(row.get(2)) else np.nan,
                    'population_density': float(row[3]) if _is_num(row.get(3)) else np.nan,
                })
        elif lv == 1 and county:
            records.append({
                'county_zh': county,
                'town_zh': _normalize_xls_name(n),
                'population': int(float(row[1])),
                'land_area_km2': float(row[2]) if _is_num(row.get(2)) else np.nan,
                'population_density': float(row[3]) if _is_num(row.get(3)) else np.nan,
            })
    return pd.DataFrame(records)


df_pop = _load_pop_density_xls(XLS_PATH, sheet='109')
print('Population XLS shape:', df_pop.shape)
print('Missing values:\n', df_pop.isnull().sum())
df_pop.head()

## Step 2 — Load Age Structure

Reads 22 county xlsx files from `民國109年常住人口之年齡結構/`.

Age bands available: <15, 15-24, 25-34, 35-44, 45-54, 55-64, 65+, average age.
Approximations applied for DeepSolar 5-year bands using uniform distribution assumption.

In [ ]:
def _load_age(age_dir):
    """
    Parse per-county age structure xlsx files.

    Columns (0-indexed): 1=name, 2=total, 3=<15, 4=15-24, 5=25-34,
    6=35-44, 7=blank, 8=45-54, 9=55-64, 10=65+, 11=avg age.
    """
    records, county = [], None
    pattern = str(age_dir / '*常住人口之年齡結構.xlsx')
    for path in sorted(glob(pattern)):
        df = pd.read_excel(path, header=None)
        for _, row in df.iterrows():
            n = row.get(1, np.nan)
            lv = _lead_spaces(n)
            if not _is_num(row.get(2)):
                continue
            total = float(row[2])
            if total == 0:
                continue
            if lv == 1:
                county = _normalize_name(n)
            elif lv == 2 and county:
                u15   = float(row[3])  if _is_num(row.get(3))  else 0.0
                a1524 = float(row[4])  if _is_num(row.get(4))  else 0.0
                a3544 = float(row[6])  if _is_num(row.get(6))  else 0.0
                a4554 = float(row[8])  if _is_num(row.get(8))  else 0.0
                a5564 = float(row[9])  if _is_num(row.get(9))  else 0.0
                a65p  = float(row[10]) if _is_num(row.get(10)) else 0.0
                avg_a = float(row[11]) if _is_num(row.get(11)) else np.nan
                records.append({
                    'county_zh': county,
                    'town_zh': _normalize_name(n),
                    'age_35_44_rate':        a3544 / total,
                    'age_45_54_rate':        a4554 / total,
                    'age_55_64_rate':        a5564 / total,
                    'age_median':            avg_a,
                    'age_5_9_rate':          u15   / total * (5 / 15),
                    'age_10_14_rate':        u15   / total * (5 / 15),
                    'age_15_17_rate':        a1524 / total * (3 / 10),
                    'age_65_74_rate':        a65p  / total * 0.60,
                    'age_more_than_85_rate': a65p  / total * 0.08,
                })
    return pd.DataFrame(records)


df_age = _load_age(AGE_DIR)
print('Age structure shape:', df_age.shape)
print('Missing values:\n', df_age.isnull().sum())
df_age.head()

## Step 3 — Load Household Size

Reads 22 county xlsx files from `民國109年住戶數、常住人口數及平均每戶人口數/`.

In [ ]:
def _load_household(hh_dir):
    """
    Parse per-county household xlsx files.

    Columns (0-indexed): 1=name, 2=total households, 4=avg persons per household.
    """
    records, county = [], None
    pattern = str(hh_dir / '*.xlsx')
    for path in sorted(glob(pattern)):
        df = pd.read_excel(path, header=None)
        for _, row in df.iterrows():
            n = row.get(1, np.nan)
            lv = _lead_spaces(n)
            if not _is_num(row.get(2)):
                continue
            if lv == 1:
                county = _normalize_name(n)
            elif lv == 2 and county:
                records.append({
                    'county_zh': county,
                    'town_zh': _normalize_name(n),
                    'average_household_size': float(row[4]) if _is_num(row.get(4)) else np.nan,
                })
    return pd.DataFrame(records)


df_hh = _load_household(HH_DIR)
print('Household shape:', df_hh.shape)
print('Missing values:\n', df_hh.isnull().sum())
df_hh.head()

## Step 4 — Join & Align to TOWNCODE Index

Merge the three tables on `(county_zh, town_zh)`, then left-join to the
climate TOWNCODE index to ensure all 368 townships are present.

In [ ]:
df_climate = pd.read_csv(CLIMATE_CSV)
df_climate['county_zh'] = df_climate['COUNTYNAME'].str.replace('台', '臺')
df_climate['town_zh']   = df_climate['TOWNNAME'].str.replace('台', '臺')

base = (
    df_pop
    .merge(df_age, on=['county_zh', 'town_zh'], how='outer')
    .merge(df_hh,  on=['county_zh', 'town_zh'], how='outer')
)

result = (
    df_climate[['TOWNCODE', 'COUNTYNAME', 'TOWNNAME', 'county_zh', 'town_zh']]
    .merge(base, on=['county_zh', 'town_zh'], how='left')
    .drop(columns=['county_zh', 'town_zh'])
)

print('Joined shape:', result.shape)
print('Missing values before total_area:')
print(result.isnull().sum())
result.head()

## Step 5 — Add `total_area` (km² → sq. miles)

US DeepSolar `total_area` is Land + Water Area in **sq. miles**.  
Taiwan `land_area_km2` is land area only in **km²**.  
Conversion: 1 sq. mile = 2.58999 km²

The water-area difference is negligible for Taiwan townships (water bodies
typically < 1% of total area) so `land_area_km2 / 2.58999` is an acceptable proxy.

In [ ]:
# land_area_km2 (km²) → total_area (sq. miles); 1 sq. mile = 2.58999 km²
result['total_area'] = result['land_area_km2'] / 2.58999

print('total_area added.')
print(f'  min  : {result.total_area.min():.3f} sq. miles')
print(f'  mean : {result.total_area.mean():.3f} sq. miles')
print(f'  max  : {result.total_area.max():.3f} sq. miles')

## Step 6 — Select Columns & Validate

In [ ]:
cols = [
    'TOWNCODE', 'COUNTYNAME', 'TOWNNAME',
    'population', 'population_density', 'land_area_km2', 'total_area',
    'age_5_9_rate', 'age_10_14_rate', 'age_15_17_rate',
    'age_35_44_rate', 'age_45_54_rate', 'age_55_64_rate',
    'age_65_74_rate', 'age_more_than_85_rate', 'age_median',
    'average_household_size',
]
result = result.reindex(columns=cols)

print('Final shape:', result.shape, '  (expected: 368 × 17)')
print('\nMissing values:')
print(result.isnull().sum())
print()

# Spot-checks
print('population_density max (expected 永和區 ~38,000):', result.population_density.max())
print('population_density min (expected 桃源區 ~4.5):', result.population_density.min())
print('age_median max (expected 龍崎區 ~55):', result.age_median.max())
print('age_median min (expected 東引鄉 ~32):', result.age_median.min())
print('average_household_size mean (expected ~2.98):', result.average_household_size.mean().round(2))
print('total_area min (smallest township):', result.total_area.min().round(3), 'sq. miles')
print('total_area max (largest township):', result.total_area.max().round(3), 'sq. miles')

result.head()

## Step 7 — Save

In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

size_kb = OUT_PATH.stat().st_size / 1024
print(f'Saved: {OUT_PATH}  ({size_kb:.0f} KB)')
print(f'Shape: {result.shape}')
print()
print('Columns:', result.columns.tolist())